In [ ]:
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from tqdm import tqdm
import yaml
import random

%matplotlib inline

In [ ]:
ORIGINAL_DATASET_FOLDER = 'data/'
PATCHES_DATASET_FOLDER = 'data_patches/'

In [ ]:
# Original image dimensions
IMG_W = 1920
IMG_H = 1080

# Patch dimensions
PATCH_SIZE = 640

# Patch grid: 4 columns x 2 rows
N_COLS = 4
N_ROWS = 2

# Compute starting positions for each patch
# Horizontal stride: (1920 - 640) / 3 = 426.67 -> rounded positions
X_STARTS = [round(i * (IMG_W - PATCH_SIZE) / (N_COLS - 1)) for i in range(N_COLS)]  # [0, 427, 853, 1280]
Y_STARTS = [round(i * (IMG_H - PATCH_SIZE) / (N_ROWS - 1)) for i in range(N_ROWS)]  # [0, 440]

# Precomputed patch origins: (row, col, x_start, y_start)
PATCH_ORIGINS = [
    (row, col, x, y)
    for row, y in enumerate(Y_STARTS)
    for col, x in enumerate(X_STARTS)
]

# Filtering thresholds
MIN_VISIBILITY = 0.2   # clipped box must retain 20% of original area
MIN_CLIPPED_PX = 2     # minimum pixel dimension after clipping

CLASS_NAMES = ['green', 'off', 'red', 'wait_on', 'yellow']

print(f'X starts: {X_STARTS}')
print(f'Y starts: {Y_STARTS}')
print(f'Patches per image: {len(PATCH_ORIGINS)}')

In [ ]:
def clip_box_to_patch(class_id, abs_cx, abs_cy, abs_w, abs_h, patch_x, patch_y):
    """ Clip an absolute-pixel bounding box to a patch and return YOLO-normalized coords.

    Args:
        class_id: integer class ID
        abs_cx, abs_cy, abs_w, abs_h: bounding box in absolute pixels (center format)
        patch_x, patch_y: top-left corner of the patch in the original image

    Returns:
        (class_id, norm_cx, norm_cy, norm_w, norm_h) or None if box is too small/invisible
    """
    # Convert center-format to corner-format
    x1 = abs_cx - abs_w / 2
    y1 = abs_cy - abs_h / 2
    x2 = abs_cx + abs_w / 2
    y2 = abs_cy + abs_h / 2

    # Patch boundaries
    px1 = patch_x
    py1 = patch_y
    px2 = patch_x + PATCH_SIZE
    py2 = patch_y + PATCH_SIZE

    # Compute intersection
    ix1 = max(x1, px1)
    iy1 = max(y1, py1)
    ix2 = min(x2, px2)
    iy2 = min(y2, py2)

    # Check for valid intersection
    if ix1 >= ix2 or iy1 >= iy2:
        return None

    clipped_w = ix2 - ix1
    clipped_h = iy2 - iy1

    # Check minimum pixel size
    if clipped_w < MIN_CLIPPED_PX or clipped_h < MIN_CLIPPED_PX:
        return None

    # Check visibility ratio
    original_area = abs_w * abs_h
    clipped_area = clipped_w * clipped_h
    if original_area > 0 and (clipped_area / original_area) < MIN_VISIBILITY:
        return None

    # Convert to patch-local normalized YOLO coordinates
    local_cx = ((ix1 + ix2) / 2 - patch_x) / PATCH_SIZE
    local_cy = ((iy1 + iy2) / 2 - patch_y) / PATCH_SIZE
    local_w = clipped_w / PATCH_SIZE
    local_h = clipped_h / PATCH_SIZE

    return class_id, local_cx, local_cy, local_w, local_h

In [ ]:
def parse_yolo_label_file(label_path):
    """ Read a YOLO label file and convert normalized coords to absolute pixels.

    Args:
        label_path: Path to the .txt label file

    Returns:
        List of (class_id, abs_cx, abs_cy, abs_w, abs_h) tuples in pixels
    """
    boxes = []
    if not os.path.exists(label_path):
        return boxes

    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            class_id = int(parts[0])
            cx = float(parts[1]) * IMG_W
            cy = float(parts[2]) * IMG_H
            w = float(parts[3]) * IMG_W
            h = float(parts[4]) * IMG_H
            boxes.append((class_id, cx, cy, w, h))

    return boxes

In [ ]:
def write_yolo_label_file(label_path, boxes):
    """ Write a list of YOLO-format bounding boxes to a label file.

    Args:
        label_path: Path to the output .txt file
        boxes: List of (class_id, cx, cy, w, h) tuples (normalized to [0,1])
    """
    with open(label_path, 'w') as f:
        for class_id, cx, cy, w, h in boxes:
            f.write(f'{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

In [ ]:
def process_split(split_name, input_base, output_base):
    """ Process one dataset split (train or val) into patches.

    Args:
        split_name: 'train' or 'val'
        input_base: path to original dataset folder (e.g. 'data/')
        output_base: path to output patches folder (e.g. 'data_patches/')
    """
    input_img_dir = Path(input_base) / split_name / 'images'
    input_lbl_dir = Path(input_base) / split_name / 'labels'
    output_img_dir = Path(output_base) / split_name / 'images'
    output_lbl_dir = Path(output_base) / split_name / 'labels'

    # Create output directories
    output_img_dir.mkdir(parents=True, exist_ok=True)
    output_lbl_dir.mkdir(parents=True, exist_ok=True)

    image_files = sorted(input_img_dir.glob('*.jpg'))
    total_input_boxes = 0
    total_output_boxes = 0
    empty_patches = 0

    for img_path in tqdm(image_files, desc=f'Processing {split_name}'):
        stem = img_path.stem

        # Read image once
        img = cv2.imread(str(img_path))
        if img is None:
            print(f'Warning: could not read {img_path}')
            continue

        # Parse label file
        label_path = input_lbl_dir / f'{stem}.txt'
        abs_boxes = parse_yolo_label_file(str(label_path))
        total_input_boxes += len(abs_boxes)

        # Process each patch
        for row, col, px, py in PATCH_ORIGINS:
            # Crop patch via numpy slicing
            patch = img[py:py + PATCH_SIZE, px:px + PATCH_SIZE]

            # Clip all bounding boxes to this patch
            patch_boxes = []
            for class_id, abs_cx, abs_cy, abs_w, abs_h in abs_boxes:
                result = clip_box_to_patch(class_id, abs_cx, abs_cy, abs_w, abs_h, px, py)
                if result is not None:
                    patch_boxes.append(result)

            total_output_boxes += len(patch_boxes)
            if not patch_boxes:
                empty_patches += 1

            # Save patch image and label
            patch_name = f'{stem}_r{row}c{col}'
            cv2.imwrite(str(output_img_dir / f'{patch_name}.jpg'), patch)
            write_yolo_label_file(str(output_lbl_dir / f'{patch_name}.txt'), patch_boxes)

    n_patches = len(image_files) * len(PATCH_ORIGINS)
    print(f'\n{split_name}: {len(image_files)} images -> {n_patches} patches')
    print(f'  Input boxes:  {total_input_boxes}')
    print(f'  Output boxes: {total_output_boxes}')
    print(f'  Empty patches: {empty_patches} / {n_patches}')

In [ ]:
def create_data_yaml(output_base):
    """ Write data.yaml for the patches dataset. """
    data = {
        'train': '../train/images',
        'val': '../val/images',
        'nc': len(CLASS_NAMES),
        'names': CLASS_NAMES,
    }
    yaml_path = Path(output_base) / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False)
    print(f'Written {yaml_path}')

In [ ]:
def visualize_patches(original_dir, patches_dir):
    """ Pick a random val image with boxes and visualize original + patches. """
    # Pick a random val image that has bounding boxes
    val_lbl_dir = Path(original_dir) / 'val' / 'labels'
    val_img_dir = Path(original_dir) / 'val' / 'images'
    patch_img_dir = Path(patches_dir) / 'val' / 'images'
    patch_lbl_dir = Path(patches_dir) / 'val' / 'labels'

    # Find label files with at least one box
    non_empty_labels = [p for p in sorted(val_lbl_dir.glob('*.txt')) if p.stat().st_size > 0]
    random.seed(42)
    chosen_label = random.choice(non_empty_labels)
    stem = chosen_label.stem

    # Load original image and boxes
    orig_img = cv2.cvtColor(cv2.imread(str(val_img_dir / f'{stem}.jpg')), cv2.COLOR_BGR2RGB)
    abs_boxes = parse_yolo_label_file(str(chosen_label))

    # --- Plot original image with boxes ---
    fig, axes = plt.subplots(1, 1, figsize=(16, 9))
    axes.imshow(orig_img)
    axes.set_title(f'Original: {stem}', fontsize=10)
    for cls_id, cx, cy, w, h in abs_boxes:
        rect = patches.Rectangle((cx - w/2, cy - h/2), w, h, linewidth=2, edgecolor='lime', facecolor='none')
        axes.add_patch(rect)
        axes.text(cx - w/2, cy - h/2 - 2, CLASS_NAMES[cls_id], color='lime', fontsize=8, fontweight='bold')
    # Draw patch grid
    for row, col, px, py in PATCH_ORIGINS:
        rect = patches.Rectangle((px, py), PATCH_SIZE, PATCH_SIZE, linewidth=1, edgecolor='cyan', facecolor='none', linestyle='--')
        axes.add_patch(rect)
    axes.set_xlim(0, IMG_W)
    axes.set_ylim(IMG_H, 0)
    plt.tight_layout()
    plt.show()

    # --- Plot 2x4 grid of patches with clipped boxes ---
    fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(16, 8))
    for row, col, px, py in PATCH_ORIGINS:
        ax = axes[row, col]
        patch_name = f'{stem}_r{row}c{col}'
        patch_img = cv2.cvtColor(cv2.imread(str(patch_img_dir / f'{patch_name}.jpg')), cv2.COLOR_BGR2RGB)
        ax.imshow(patch_img)
        ax.set_title(f'r{row}c{col} ({px},{py})', fontsize=9)

        # Load and draw patch labels
        patch_label_path = patch_lbl_dir / f'{patch_name}.txt'
        if patch_label_path.exists():
            with open(patch_label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls_id = int(parts[0])
                    ncx, ncy, nw, nh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    # Convert normalized to pixel coords within patch
                    bx = (ncx - nw/2) * PATCH_SIZE
                    by = (ncy - nh/2) * PATCH_SIZE
                    bw = nw * PATCH_SIZE
                    bh = nh * PATCH_SIZE
                    rect = patches.Rectangle((bx, by), bw, bh, linewidth=2, edgecolor='lime', facecolor='none')
                    ax.add_patch(rect)
                    ax.text(bx, by - 2, CLASS_NAMES[cls_id], color='lime', fontsize=7, fontweight='bold')

        ax.set_xlim(0, PATCH_SIZE)
        ax.set_ylim(PATCH_SIZE, 0)
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle(f'Patches for: {stem}', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
def print_statistics(patches_dir):
    """ Print a summary of the patch dataset statistics. """
    print('=' * 50)
    print('PATCH DATASET STATISTICS')
    print('=' * 50)

    for split in ['train', 'val']:
        img_dir = Path(patches_dir) / split / 'images'
        lbl_dir = Path(patches_dir) / split / 'labels'

        n_images = len(list(img_dir.glob('*.jpg')))
        label_files = list(lbl_dir.glob('*.txt'))
        n_labels = len(label_files)
        n_non_empty = sum(1 for lf in label_files if lf.stat().st_size > 0)

        print(f'\n{split}:')
        print(f'  Images:           {n_images}')
        print(f'  Label files:      {n_labels}')
        print(f'  Non-empty labels: {n_non_empty}')
        print(f'  Empty labels:     {n_labels - n_non_empty}')

In [ ]:
process_split('train', ORIGINAL_DATASET_FOLDER, PATCHES_DATASET_FOLDER)
process_split('val', ORIGINAL_DATASET_FOLDER, PATCHES_DATASET_FOLDER)
create_data_yaml(PATCHES_DATASET_FOLDER)
visualize_patches(ORIGINAL_DATASET_FOLDER, PATCHES_DATASET_FOLDER)
print_statistics(PATCHES_DATASET_FOLDER)